In [ ]:
# Idea: the likelihood is a linear model: prediction  = matrix x params - thus closed form sol - dont need to sample.

import numpy as np
import yaml
import copy
from cobaya import get_model
import matplotlib.pyplot as plt
import os

BPE_CHAINS = os.environ.get('BPE_CHAINS', '/path/to/chains')
CFGDIR = '.' # configs alongside the notebook




def build_like(base_yaml, kmax=None, order=None):
    """
    function that builds the likelihood object at a given kmax / order through cobaya.get_model().

    """
    info = yaml.safe_load(open(f'{CFGDIR}/{base_yaml}')) # load specific yaml file of interest
    info = copy.deepcopy(info)
    # cobaya complains about these when you only want a model, not a run
    # for k in ['sampler', 'output', 'resume', 'force', 'debug', 'stop_at_error']:
        # info.pop(k, None)

    print(info)
    info.pop("sampler")

    lk = info['likelihood']['bUClike.galCMBLike']
    if kmax  is not None: lk['kmax']  = kmax
    if order is not None: lk['order'] = order

    # cobaya constructs everything and calls the initialise constuctor for a given yaml instance.
    model = get_model(info) 
    return model.likelihood['bUClike.galCMBLike']


def solve_analytic(like):
    """
    Generalised least squares on the full template matrix.

    • _build_template_mat is called with no parameters. 
    • That works because mode == "default": update_CIB_tracers only reads params_values in the "dbeta"/"w" modes, and update_gc_templates doesn't read them at all. 
    • So there's no best-fit dict to unpack and nothing to set to zero - which qsidesteps the missing .minimum file entirely.

    """
    T = like._build_template_mat() # The template matrix: one row per data point (1218), one column per parameter.
    d = like.ps_vec # the measured Cls, already filtered
    invC = like.invcov # inverse covariance, computed in initialize()

    Ctheta_inv = T.T @ invC @ T # (ctheta_inv - saras appendix A)
    Ctheta = np.linalg.inv(Ctheta_inv)
    theta = Ctheta @ T.T @ invC @ d # also defined in saras appendix.

    # goodness of fit test
    res = d - T @ theta # resdiual
    chi2 = float(res.T @ invC @ res) # 
    dof = len(d) - T.shape[1] # T.shape[1] = number of cols = number of params.   

    return theta, Ctheta, chi2, dof, T.shape, d


def derive(theta, Ctheta, clust):
    """
    bg = sqrt(gg),  bPe = gy/sqrt(gg),  brhoSFR = gc/sqrt(gg)

    Errors by propagation - f = y/sqrt(x)  -  df/dx = -y/(2 x^1.5),  df/dy = 1/sqrt(x)
    """

    n = len(clust) # should be 6 for these files (BGS+LRG)
    out = {}
    for i, t in enumerate(clust): # for (index, tracer (BGS+LRGS))
        
                  # gg.    # gPe.     # grhoSFR - each array of 6 params or so.
        x, y, z = theta[i], theta[n+i], theta[2*n+i] # The column ordering comes from _build_template_mat
        # comes from in galCMBLike - par_id = 3*len(clust)*idx_temp + idx_bin + {0,1,2}*len(clust)
        # idx_temp = 0 = the plain mm template.       
        # first 6 cols are gg params - 6-11 are gY params and gCIB are 12-17, hence the thetas - look for as par_id in galCMBlike

        # theta vector is 3*ncluster long.

        vx, vy, vz = Ctheta[i,i], Ctheta[n+i,n+i], Ctheta[2*n+i,2*n+i] # Ctheta is cov mat - variances
        cxy, cxz   = Ctheta[i,n+i], Ctheta[i,2*n+i] # covarinces. these specfifically for error prop

        # error prop
        dfx = -y/(2*x**1.5); dfy = 1/np.sqrt(x)
        dgx = -z/(2*x**1.5); dgy = 1/np.sqrt(x)


        out[t] = dict(
            bg = np.sqrt(x),
            bg_err = np.sqrt(vx/(4*x)),
            bPe = y/np.sqrt(x),
            bPe_err = np.sqrt(abs(dfx**2*vx + dfy**2*vy + 2*dfx*dfy*cxy)),
            sfr = z/np.sqrt(x),
            sfr_err = np.sqrt(abs(dgx**2*vx + dgy**2*vz + 2*dgx*dgy*cxz)),
        )
    return out



# sweep the configurations

configs = [
    ('0.3 k^2', 'baseline_v2.yaml', 0.3, 2),
    ('0.2 k^2', 'baseline_v2.yaml', 0.2, 2),
    ('0.3 k^4', 'baseline_v2.yaml', 0.3, 4)
]

"""
configs_stable = [
    ('0.15 k^0', 'baseline_v2.yaml', 0.15, 0),
    ('0.3 k^2', 'baseline_v2.yaml', 0.3, 2),
    ('0.5 k^4', 'baseline_v2.yaml', 0.5, 4)
]
"""
results = {}
for label, yml, kmax, order in configs:
    print(f"{label}")
    like  = build_like(yml, kmax=kmax, order=order)
    clust = like.fields_to_use['clustering'] # 6 - BGS0,1 and LRG0,1,2,3.
    theta, Ctheta, chi2, dof, shape, d = solve_analytic(like)
    print(len(d))
    results[label] = derive(theta, Ctheta, clust)
    print(f"data vector {shape[0]}, {shape[1]} params, "
          f"chi2/dof = {chi2:.1f}/{dof} = {chi2/dof:.3f}")



# the comparison table - this is the answer to "sampling or real?"

tracers = list(results[configs[0][0]].keys())

for key, scale, name in [('bPe', 1e3, 'bPe [meV/cm^3]'),
                         ('bg',  1.0, 'bg'),
                         ('sfr', 1.0, 'brhoSFR')]:
    print(f"{name} : analytic")
    print(f"{'tracer':14} " + " ".join(f"{c[0]:>17}" for c in configs))
    for t in tracers:
        row = []
        for c in configs:
            r = results[c[0]][t]
            row.append(f"{scale*r[key]:8.3f}±{scale*r[key+'_err']:7.3f}")
        print(f"{t:14} " + " ".join(row))



# analytic vs MCMC.

from getdist.mcsamples import loadMCSamples

mcmc = {'0.3 k^2': 'baseline_v2', '0.2 k^2': 'kmax02_v2', '0.3 k^4': 'baseline_v2_order4'}

print(f"bPe : analytic vs MCMC")
print(f"{'tracer':14} " + " ".join(f"{k+' anl':>15} {k+' mcmc':>15}" for k in mcmc))
for t in tracers:
    row = []
    for lab, chain in mcmc.items():
        s = loadMCSamples(f'{BPE_CHAINS}/{chain}', settings={'ignore_rows': 0.3})
        a_ = results[lab][t]
        row.append(f"{1e3*a_['bPe']:7.3f}±{1e3*a_['bPe_err']:6.3f}")
        row.append(f"{1e3*s.mean(f'bPe_{t}'):7.3f}±{1e3*s.std(f'bPe_{t}'):6.3f}")
    print(f"{t:14} " + " ".join(row))   



In [ ]:
def cov2corr(C):
    """
    R_ij = C_ij / sqrt(C_ii C_jj)
    (from SHP.)
    """

    s = np.sqrt(np.diag(C))
    return C / np.outer(s, s) # outer product of two vectors.


def plot_corr(C, labels, title):

    """
    function defining the plotting to match adriens type of formatting.
    """

    R = cov2corr(C)
    
    _, ax = plt.subplots(figsize=(0.55*len(labels)+3,)*2)
    im = ax.imshow(R, cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=90, fontsize=7) # set ticks on graph - rotate x labels so dont overlap.
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=7)

    # loop over 2D grid 
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, f"{R[i,j]:.2f}", ha='center', va='center', # write the corr value in centre of cell.
                    fontsize=5, color='k' if abs(R[i,j]) < 0.6 else 'w') # white or black text based on corrleation colour - purely cosmetic most of this.
    ax.set_title(title, fontsize=9)
    return im


# just physical params for now. 
def phys_block(Ctheta, clust):

    """
    function defining where the physical params are and giving those and index and a label.

    """
    n = len(clust)
    idx = np.arange(3*n) # first 3*nclust columns are the physical params.
    lab = ([f'bgg_{t.split("__")[-1]}' for t in clust] +
           [f'bgU_{t.split("__")[-1]}' for t in clust] +
           [f'bgC_{t.split("__")[-1]}' for t in clust])
    return Ctheta[np.ix_(idx, idx)], lab



for label, yml, kmax, order in configs:
    like = build_like(yml, kmax=kmax, order=order)
    clust = like.fields_to_use['clustering']
    _, Ctheta, _, _, _, _ = solve_analytic(like)
    Cb, lab = phys_block(Ctheta, clust)
    im = plot_corr(Cb, lab, label)
    plt.colorbar(im, shrink=0.7, label='correlation')



In [ ]:
def cov2corr(C):
    """
    R_ij = C_ij / sqrt(C_ii C_jj)
    (from SHP.)
    """

    s = np.sqrt(np.diag(C))
    return C / np.outer(s, s) # outer product of two vectors.


def plot_corr(C, labels, title):

    """
    function defining the plotting to match adriens type of formatting.
    """

    R = cov2corr(C)
    
    _, ax = plt.subplots(figsize=(0.28*len(labels)+4,)*2)
    im = ax.imshow(R, cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=90, fontsize=6) # set ticks on graph - rotate x labels so dont overlap.
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=6)

    # loop over 2D grid 
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, f"{R[i,j]:.2f}", ha='center', va='center', # write the corr value in centre of cell.
                    fontsize=4, color='k' if abs(R[i,j]) < 0.6 else 'w') # white or black text based on corrleation colour - purely cosmetic most of this.
    ax.set_title(title, fontsize=9)
    return im


# all params now instead of just physical - a full inspection.
def all_params(Ctheta, clust, order):

    """
    function labelling every parameter in the template matrix, in the order
    the columns are written.

    """
    ntemp = order//2 + 1
    tnames = ["mm", "k2", "k4"][:ntemp] + ["N"] # noise block comes last
    short = [t.split("__")[-1] for t in clust]

    lab = []
    for tn in tnames:
        for bt in ["bgg", "bgU", "bgC"]:
            for s in short:
                lab.append(f"{bt}.{tn}.{s}")

    return Ctheta, lab



# one figure per config - 54x54 (or 72x72 at order 4) needs the room
for label, yml, kmax, order in configs:
    like = build_like(yml, kmax=kmax, order=order)
    clust = like.fields_to_use['clustering']
    _, Ctheta, _, _, _, _ = solve_analytic(like)
    Cb, lab = all_params(Ctheta, clust, order)
    im = plot_corr(Cb, lab, f"{label}  ({len(lab)} params)")
    plt.colorbar(im, shrink=0.7, label='correlation')
    plt.tight_layout()


In [ ]:
def param_index(block, template, nclust):
    """
    function defining the parameter id eqn - same as adriens code - used here to get params of interets.

    For reference, Adriens: par_id = (3 * len(clust) * idx_temp + idx_bin + 0 * len(clust))
    """

    # temp = {plain (mm): 0, k^2: 1, K^4: 2} - moving one temp along jumps 3*nclust along
    # block = which spectrum, 0 = gg, 1 = gtSZ, 2- gCIB. - moving one block jumps nclust along. 
  
    return 3*nclust*template + block*nclust  


def tracer_block(Ctheta, clust, order):
    """
    The 3x(ntemp+1) parameters belonging to one tracer, and their labels.
    Adrien's point: bin-to-bin correlations are small, so each tracer's own
    block is the informative thing to look at.
    """

    n = len(clust)
    ntemp = order//2 + 1 # how many k^n templates exist (order 2, then 2 - mm and k^2)
    snames = ["gg", "gU", "gC"] # spec names

    idx, lab = [], []

    # loop over a template block (mm, k^2, k^4)
    for t in range(ntemp):

        # within each template block, loop over spectrum types.
        for s, sn in enumerate(snames):

            # append the param id to list
            idx.append(param_index(s, t, n))

            # append name to list to indetify the index.
            lab.append((["bg", "A", "B"][t]) + sn)

    # the same but param_idx at ntemp - noise terms.
    for s, sn in enumerate(snames): # noise block
        idx.append(param_index(s, ntemp, n))
        lab.append(f"N{sn}")

    
    idx = np.array(idx) # make numpy array of idicies
    return Ctheta[np.ix_(idx, idx)], lab # np.ix_ selects the sub matrix, those rows and columns - not elements.



# 1) eigenvalue spectra (Davids suggestion)

covs = {} # create covaraince dict
fig, ax = plt.subplots(figsize=(8, 5.5))


# for each yaml file run
for label, yml, kmax, order in configs:

    # get likelihood
    like = build_like(yml, kmax=kmax, order=order)

    # select all tracers - BGS and LRGS
    clust = like.fields_to_use['clustering']

    # solve analytical solution
    out = solve_analytic(like)
    
    # get cov matrix from analytical solution.
    Ctheta = out[1]

    # for each yaml, assign the cov, tracers and order to dict.
    covs[label] = (Ctheta, clust, order)

    # returns real eigenvalues in asceding order
    ev = np.linalg.eigvalsh(Ctheta)[::-1] # select in desceding order
    

    neg = ev < 0 # create boolean where any are negative.
    
    # plot absolute eigen values
    ax.plot(np.arange(len(ev)), np.abs(ev), '-o', ms=3, label=label)
    if neg.any():
        ax.plot(np.arange(len(ev))[neg], np.abs(ev[neg]), 'x', ms=9,
                color='r', mew=2)

    print(f"{label:10}  n={len(ev):3d}  "
          f"max={ev[0]:10.3e}  min={ev[-1]:10.3e}  "
          f"cond={abs(ev[0]/ev[-1]):9.2e}  n_negative={neg.sum()}") # single number showing how degenerate the system is, condition - largest /smallest

ax.set_yscale('log')
ax.set_xlabel('eigenvalue rank (largest first)')
ax.set_ylabel(r'eigenvalues of the parameter covariance')
ax.set_title('red x = negative eigenvalue')
ax.legend(fontsize=8)
plt.tight_layout()



# 2) per-tracer blocks: bg vs the k^2 and noise amplitudes (Adriens suggestion)


# loop over eachh yaml.
for label in covs:


    Ctheta, clust, order = covs[label]
    

    ncol = min(3, len(clust)) # number of columns
    nrow = int(np.ceil(len(clust)/ncol)) # number of rows.

    # fig and axes
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.4*ncol, 4.4*nrow))
    axes = np.atleast_1d(axes).ravel() # ravel falttens 2d array of axes so can index with a single counter.

    # loop over tracers, BGS and LRGS    
    for i, t in enumerate(clust):

        
        Cb, lab = tracer_block(Ctheta, clust, order) # return cov matrix of all rows and col of idx,idx - sample of covariance matrix of interest.

        R = cov2corr(Cb) # correlation matrix
        ax_ = axes[i]
        ax_.imshow(R, cmap='RdBu_r', vmin=-1, vmax=1)
        ax_.set_xticks(range(len(lab))); ax_.set_xticklabels(lab, rotation=90, fontsize=7)
        ax_.set_yticks(range(len(lab))); ax_.set_yticklabels(lab, fontsize=7)
        for a in range(len(lab)):
            for b in range(len(lab)):
                ax_.text(b, a, f"{R[a,b]:.2f}", ha='center', va='center',
                         fontsize=6, color='k' if abs(R[a,b]) < 0.6 else 'w')
        ax_.set_title(t, fontsize=9)
    for j in range(len(clust), len(axes)):
        axes[j].axis('off') # blancks any unused panels
    fig.suptitle(f"{label} - per-tracer parameter block", fontsize=11)
    plt.tight_layout()



# the one number per tracer that matters: how correlated is bgU with the nuisances it gets marginalised against?

pairs = [("bgU", "AgU"), ("bgU", "NgU"), ("bgg", "Agg"), ("bgg", "Ngg"),
         ("bgg", "bgU")] # specific pairs adrien was interested in.

# loop over pairs
for pa, pb in pairs:
    
    
    print(f" corr({pa}, {pb}) ")
    print(f"{'tracer':14} " + " ".join(f"{c[0]:>10}" for c in configs))
    
    # loop over covarainces for each confif file.
    for i, t in enumerate(covs[configs[0][0]][1]):

        row = []

        # for elements in configs.
        for label, *_ in configs:
            Ctheta, clust, order = covs[label]
            Cb, lab = tracer_block(Ctheta, clust, order)
            R = cov2corr(Cb)

            # get values of interest, append - other
            if pa in lab and pb in lab:
                row.append(f"{R[lab.index(pa), lab.index(pb)]:10.3f}")
            else:
                row.append(f"{'-':>10}")
        

# Quaia Value Solve

In [ ]:


like2 = build_like('quaia_v3.yaml', kmax=0.15, order=0)
theta2, Ctheta2, chi22, dof2, shape2, _ = solve_analytic(like2)
clust2 = like2.fields_to_use['clustering']
analytic2 = derive(theta2, Ctheta2, clust2)


print(f"{'param':22} {'analytic':>18}") 
for t in clust2:
    for kind, scale in [('bPe', 1e3), ('bg', 1.0), ('sfr', 1.0)]:
        a_mean = scale * analytic2[t][kind]
        a_err  = scale * analytic2[t][kind + '_err']


        name = f"{'brhoSFR' if kind=='sfr' else kind}_{t}"
        print(f"{name:22} {a_mean:8.4f}±{a_err:6.4f} ")
